In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import re, sys, subprocess, gc, random
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, set_seed
from accelerate import infer_auto_device_map as iadm

set_seed(199)
model_id = "/kaggle/input/deepseek-math/pytorch/deepseek-math-7b-rl/1"
config = AutoConfig.from_pretrained(model_id)
config.gradient_checkpointing = True
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto", trust_remote_code=True, config=config)
#device_map = iadm(model, max_memory={0: "15GiB", 1: "15GiB", "cpu": "25GiB"})

2025-02-21 18:58:30.562172: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-21 18:58:30.562295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-21 18:58:30.706261: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:

pipeline = transformers.pipeline("text-generation", model=model, tokenizer=tokenizer, torch_dtype='auto', device_map="auto")
torch.backends.cuda.enable_mem_efficient_sdp(False)

# EDA

In [5]:
import pandas as pd
from tqdm import tqdm
PRIVATE = True

df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/test.csv')
df.head()

,row_id,id,problem
0,0,000aaa,What is $1-1$?
1,1,111bbb,What is $0\times10$?
2,2,222ccc,Solve $4+x=4$ for $x$.


In [6]:
if len(df) < 5:
    df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/train.csv')
    PRIVATE = False
df.head()

,id,problem,answer
0,229ee8,"Let $k, l > 0$ be parameters. The parabola $y ...",52
1,246d26,Each of the three-digits numbers $111$ to $999...,250
2,2fc4ad,Let the `sparkle' operation on positive intege...,702
3,430b63,What is the minimum value of $5x^2+5y^2-8xy$ w...,800
4,5277ed,There exists a unique increasing geometric seq...,211


In [7]:
def naive_parse(answer):
    out = []
    start = False
    end = False
    for l in reversed(list(answer)):
        if l in '0123456789' and not end:
            start = True
            out.append(l)
        else:
            if start:
                end = True
        
    out = reversed(out)
    return ''.join(out)

In [ ]:
precode = """from scipy import *
import signal, math, scipy
def handler(signum, frame): raise Exception('Timeout Error')
signal.signal(signal.SIGALRM, handler)
signal.alarm(10)\n"""

In [8]:
import re
import sys
import subprocess


def process_output(output):
    result = output
    
    try:
        code = output.split('```')[1][7:]

        with open('code.py', 'w') as fout:
            fout.write(code)

        batcmd = 'timeout 7 ' + sys.executable + ' code.py'
        try:
            shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
            print(shell_output)
            code_output = round(float(eval(shell_output))) % 1000
        except:
            code_output = -1

        print('CODE RESULTS', code_output)
    
    except Exception as e:
        print(e)
        print('ERROR PARSING')
        code_output = -1
    
    try:
        result_output = re.findall(r'\\boxed\{(.*)\}', result)

        print('BOXED', result_output)
        if not len(result_output):
            result_output = naive_parse(result)
        else:
            result_output = result_output[-1]

        print('BOXED', result_output)
        if not len(result_output):
            result_output = -1
        
        else:
            result_output = round(float(eval(result_output))) % 1000
    
    except Exception as e:
        print(e)
        print('ERROR PARSING')
        result_output = -1
    
    return result_output, code_output

In [10]:
import re
from collections import defaultdict


tool_instruction = " The answer should be given as a non-negative modulo 1000."
tool_instruction += '\nPlease integrate natural language reasoning with programs to solve the problem above, and put your final answer within \\boxed{}.'


n_repetitions = 5 if PRIVATE else 2

total_results = []
total_answers = []

for i in tqdm(range(len(df))):
    id_ = df['id'].loc[i]
    problem = df['problem'].loc[i]
    
    messages = [
        {
            "role": "user", 
            "content": problem + tool_instruction
        }
    ]
    
    query_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
    
    results = []
    answers = []
    
    for _ in tqdm(range(n_repetitions)):
        try:
            raw_output = pipeline(
                query_prompt, 
                max_new_tokens=2048, 
                do_sample=True, 
                temperature=0.7,
                return_full_text=False
            )
            raw_output = raw_output[0]['generated_text']

            result_output, code_output = process_output(raw_output)

            torch.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            print(e)
            result_output, code_output = -1, -1
        
        results.append(result_output)
        answers.append(code_output)
    
    total_results.append(results)
    total_answers.append(answers)

  0%|          | 0/2 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


96

CODE RESULTS 96
BOXED ['96']
BOXED 96



 50%|█████     | 1/2 [02:48<02:48, 168.44s/it]Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


36

CODE RESULTS 36
BOXED ['292']
BOXED 292



  0%|          | 0/2 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


...


  0%|          | 0/2 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Traceback (most recent call last):
  File "/kaggle/working/code.py", line 13, in <module>
    result = find_f_of_100()
  File "/kaggle/working/code.py", line 7, in find_f_of_100
    solutions = solve(eq)
  File "/opt/conda/lib/python3.10/site-packages/sympy/solvers/solvers.py", line 1145, in solve
    solution = _solve(f[0], *symbols, **flags)
  File "/opt/conda/lib/python3.10/site-packages/sympy/solvers/solvers.py", line 1693, in _solve
    raise NotImplementedError('\n'.join([msg, not_impl_msg % f]))
NotImplementedError: multiple generators [f_100, floor(793/f_100)]
No algorithms are implemented to solve equation f_100 - floor(793/f_100)



CODE RESULTS -1
BOXED []
BOXED 793


 50%|█████     | 1/2 [03:46<03:46, 226.59s/it]Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


None

CODE RESULTS -1
BOXED ['987']
BOXED 987



100%|██████████| 10/10 [28:05<00:00, 168.51s/it]


In [11]:
import numpy as np
from collections import Counter

final_answers = []

for a, b in zip(total_answers, total_results):
    a = np.array(a)
    b = np.array(b)
    a[a < 0] = b[a < 0]
    
    pred = Counter(a.tolist()).most_common(2)

    ans = pred[0][0] if not pred[0][0] < 0 else pred[1][0]

    final_answers.append(ans)
    print(ans)


96
500
155
800
618
100
161
256
436
793


In [13]:
if not PRIVATE:
    df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/train.csv')
    df['model_answer'] = final_answers
    df['match'] = df.answer == df.model_answer
    print(f'{df.match.sum()} matches in {len(df)} examples')

1 matches in 10 examples


# Credits
* https://www.kaggle.com/code/olyatsimboy/aimo-zero-shot-sc-mmos-deepseekmath
*  https://www.kaggle.com/code/quan0095/more-diversity-in-output-improve-score
* https://www.kaggle.com/competitions/ai-mathematical-olympiad-prize/discussion/492578

# Planned Updates
* Include additional datasets for testing
* Add a lightweight LLM (<2.5BP) for classifying math problems
* Based on classification and parsing apply different reasoning/logic approaches for solving

# Ｈ𝐀𝑷𝑷𝓎 🇰𝗮𝘨𝘨🇱𝖎Ｎɢ 💯